FUZZY

Deklarasi Variabel Input dan Output

In [1]:
!pip install numpy scikit-fuzzy networkx scipy packaging

import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import itertools

Membership Functions

In [3]:
#Input: Skala 0.0 sampai 5.0
study_load = ctrl.Antecedent(np.arange(0, 5.1, 0.1), 'study_load')
sleep_quality = ctrl.Antecedent(np.arange(0, 5.1, 0.1), 'sleep_quality')
extracurricular = ctrl.Antecedent(np.arange(0, 5.1, 0.1), 'extracurricular')
career_concern = ctrl.Antecedent(np.arange(0, 5.1, 0.1), 'career_concern')

# Output: Skala 0.0 sampai 2.0
stress_level = ctrl.Consequent(np.arange(0, 2.1, 0.1), 'stress_level')

study_load.automf(names=['ringan', 'sedang', 'berat'])
sleep_quality.automf(names=['buruk', 'cukup', 'baik'])
extracurricular.automf(names=['pasif', 'aktif', 'sangat_aktif'])
career_concern.automf(names=['rendah', 'sedang', 'tinggi'])

stress_level.automf(names=['rendah', 'sedang', 'tinggi'])

Rule Base

In [4]:
terms_study = ['ringan', 'sedang', 'berat']         # Skor: 0, 1, 2
terms_sleep = ['baik', 'cukup', 'buruk']            # Skor: 0, 1, 2 
terms_extra = ['pasif', 'aktif', 'sangat_aktif']    # Skor: 0, 1, 2
terms_career = ['rendah', 'sedang', 'tinggi']       # Skor: 0, 1, 2

daftar_aturan = []

# Loop ini akan memutar semua 81 kemungkinan kombinasi (3x3x3x3)
for st, sl, ex, ca in itertools.product(range(3), range(3), range(3), range(3)):
    # Hitung total skor bobot keparahan (Max skor = 8, Min skor = 0)
    total_skor = st + sl + ex + ca
    
    # Logika Pakar (Expert Logic) untuk menentukan tingkat stres
    if total_skor <= 2:
        out_term = 'rendah'
    elif total_skor <= 5:
        out_term = 'sedang'
    else:
        out_term = 'tinggi'
        
    # Pembuatan Rule secara dinamis
    rule = ctrl.Rule(
        study_load[terms_study[st]] & 
        sleep_quality[terms_sleep[sl]] & 
        extracurricular[terms_extra[ex]] & 
        career_concern[terms_career[ca]], 
        stress_level[out_term]
    )
    daftar_aturan.append(rule)

Kontrol Sistem & Simulasi

In [5]:
stress_ctrl = ctrl.ControlSystem(daftar_aturan)
stress_simulasi = ctrl.ControlSystemSimulation(stress_ctrl)

Fungsi Pengujian Kasus

In [6]:
def uji_kasus(nama_kasus, load, sleep, extra, career):
    print(f"\n--- Menguji {nama_kasus} ---")
    print(f"Input -> Tugas: {load}, Tidur: {sleep}, Organisasi: {extra}, Karir: {career}")
    
    stress_simulasi.input['study_load'] = load
    stress_simulasi.input['sleep_quality'] = sleep
    stress_simulasi.input['extracurricular'] = extra
    stress_simulasi.input['career_concern'] = career
    
    # Eksekusi Mamdani & Centroid of Area
    stress_simulasi.compute()
    hasil = stress_simulasi.output['stress_level']
    
    print(f"Skor Defuzzifikasi (0-2) : {hasil:.2f}")
    
    if hasil >= 1.15:
        print("Kategori Sistem        : STRES TINGGI")
    elif hasil >= 0.85:
        print("Kategori Sistem        : STRES SEDANG")
    else:
        print("Kategori Sistem        : STRES RENDAH")
    print("-" * 35)


Uji 3 Kasus

In [7]:
print("=== SISTEM INFERENSI FUZZY MAMDANI: PREDIKSI STRES ===")
uji_kasus("Kasus 1 (Burnout Ekstrem)", load=4.5, sleep=1.0, extra=4.0, career=4.0)
uji_kasus("Kasus 2 (Mahasiswa Santai)", load=1.0, sleep=4.5, extra=1.0, career=1.0)
uji_kasus("Kasus 3 (Tekanan Menengah)", load=3.0, sleep=3.0, extra=3.0, career=3.0)

=== SISTEM INFERENSI FUZZY MAMDANI: PREDIKSI STRES ===

--- Menguji Kasus 1 (Burnout Ekstrem) ---
Input -> Tugas: 4.5, Tidur: 1.0, Organisasi: 4.0, Karir: 4.0
Skor Defuzzifikasi (0-2) : 1.18
Kategori Sistem        : STRES TINGGI
-----------------------------------

--- Menguji Kasus 2 (Mahasiswa Santai) ---
Input -> Tugas: 1.0, Tidur: 4.5, Organisasi: 1.0, Karir: 1.0
Skor Defuzzifikasi (0-2) : 0.82
Kategori Sistem        : STRES RENDAH
-----------------------------------

--- Menguji Kasus 3 (Tekanan Menengah) ---
Input -> Tugas: 3.0, Tidur: 3.0, Organisasi: 3.0, Karir: 3.0
Skor Defuzzifikasi (0-2) : 1.02
Kategori Sistem        : STRES SEDANG
-----------------------------------
